# Dependencies

In [1]:
%pip install torch "transformers>=4.50" "peft>=0.15" "bitsandbytes>=0.45" "accelerate>=1.5" "datasets>=3.5" "huggingface_hub>=0.30" "mlflow>=2.20" "sacrebleu>=2.5" "pandas>=2.2" "numpy>=2.0" "python-dotenv>=1.0" sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 116.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 92.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 75.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [mlflow]17/18 [mlflow]bytes]]]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Must run BEFORE anything initialises CUDA (restart the kernel if it already ran).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Code

## Shared split, metric and run record (Phase 2)

The next four cells are identical in all four notebooks (copied from `common/` by `tools/sync_shared_cells.py`). They give every model **the same 156 validation documents** (checked by a SHA-256 of the sorted ids), **the same metric** (sacreBLEU-identical BLEU and chrF++ with word n-grams, `tests/test_metrics.py`), write one prediction file plus one JSON run record per model into `results/`, and record the package versions and GPU (`results/env_*.json`).

In [3]:
# >>> SHARED CELL: common/split.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import hashlib
import random

ID_COL = "oare_id"
VAL_FRACTION = 0.1
SPLIT_SEED = 42
# sha256 of "\n".join(sorted(val_ids)) for the competition train.csv (1561 documents -> 156 val).
EXPECTED_VAL_SHA256 = "1939161cfcb9ff85082f20b36be2362da429f1da14f8a94f289249ffb78e1304"
EXPECTED_N_VAL = 156


def val_ids_sha256(val_ids):
    return hashlib.sha256("\n".join(sorted(map(str, val_ids))).encode("utf-8")).hexdigest()


def make_split(df, id_col=ID_COL, val_fraction=VAL_FRACTION, seed=SPLIT_SEED, check=True):
    """Return (train_df, val_df), split by document id. Rows with empty text are dropped first."""
    df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
    doc_ids = sorted(df[id_col].astype(str).unique().tolist())
    random.Random(seed).shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * val_fraction))
    val_ids = set(doc_ids[:n_val])
    is_val = df[id_col].astype(str).isin(val_ids)
    train_df = df[~is_val].reset_index(drop=True)
    val_df = df[is_val].reset_index(drop=True)
    digest = val_ids_sha256(val_ids)
    print(f"[shared split] train={len(train_df)} rows, val={len(val_df)} rows "
          f"({n_val} documents), val_ids_sha256={digest[:16]}...")
    if check:
        assert n_val == EXPECTED_N_VAL and digest == EXPECTED_VAL_SHA256, (
            "Validation split differs from the shared reference split "
            f"(n_val={n_val}, sha256={digest}). Is train.csv the original competition file?")
    assert not set(train_df[id_col].astype(str)) & val_ids, "document leakage between train and val"
    return train_df, val_df

In [4]:
# >>> SHARED CELL: common/metrics.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import math
import re
from collections import Counter

# ---------------------------------------------------------------- BLEU (13a)
_TOK_13A_RULES = [
    (re.compile(r'([\{-\~\[-\` -\&\(-\+\:-\@\/])'), r' \1 '),
    (re.compile(r'([^0-9])([\.,])'), r'\1 \2 '),
    (re.compile(r'([\.,])([^0-9])'), r' \1 \2'),
    (re.compile(r'([0-9])(-)'), r'\1 \2 '),
]


def _tokenize_13a(line):
    line = line.replace('<skipped>', '').replace('-\n', '').replace('\n', ' ')
    if '&' in line:
        line = (line.replace('&quot;', '"').replace('&amp;', '&')
                    .replace('&lt;', '<').replace('&gt;', '>'))
    line = f' {line} '
    for pattern, repl in _TOK_13A_RULES:
        line = pattern.sub(repl, line)
    return ' '.join(line.split())


def _word_ngrams(tokens, max_order):
    return Counter(tuple(tokens[i:i + n])
                   for n in range(1, max_order + 1)
                   for i in range(len(tokens) - n + 1))


def bleu_stats(hyp, ref, max_order=4):
    """Sufficient statistics of one segment: [hyp_len, ref_len, correct_1..4, total_1..4]."""
    h = _tokenize_13a(hyp.rstrip()).split()
    r = _tokenize_13a(ref.rstrip()).split()
    h_ng, r_ng = _word_ngrams(h, max_order), _word_ngrams(r, max_order)
    correct, total = [0] * max_order, [0] * max_order
    for ng, c in h_ng.items():
        total[len(ng) - 1] += c
        if ng in r_ng:
            correct[len(ng) - 1] += min(c, r_ng[ng])
    return [len(h), len(r)] + correct + total


def bleu_from_stats(stats, max_order=4):
    sys_len, ref_len = stats[0], stats[1]
    correct, total = stats[2:2 + max_order], stats[2 + max_order:]
    bp = 1.0
    if sys_len < ref_len:
        bp = math.exp(1 - ref_len / sys_len) if sys_len > 0 else 0.0
    if not any(correct):
        return 0.0
    precisions = [0.0] * max_order
    smooth = 1.0
    for n in range(max_order):
        if total[n] == 0:
            break
        if correct[n] == 0:
            smooth *= 2
            precisions[n] = 100.0 / (smooth * total[n])
        else:
            precisions[n] = 100.0 * correct[n] / total[n]
    log_sum = sum(-9999999999 if p == 0 else math.log(p) for p in precisions)
    return bp * math.exp(log_sum / max_order)


# ---------------------------------------------------------------- chrF++
_PUNCTS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')


def _chrf_words(sent):
    out = []
    for w in sent.split():
        if len(w) == 1:
            out.append(w)
        elif w[-1] in _PUNCTS:
            out += [w[:-1], w[-1]]
        elif w[0] in _PUNCTS:
            out += [w[0], w[1:]]
        else:
            out.append(w)
    return out


def _chrf_ngrams(sent, char_order, word_order):
    chars = ''.join(sent.split())
    grams = [Counter(chars[i:i + n] for i in range(len(chars) - n + 1))
             for n in range(1, char_order + 1)]
    words = _chrf_words(sent)
    grams += [Counter(' '.join(words[i:i + n]) for i in range(len(words) - n + 1))
              for n in range(1, word_order + 1)]
    return grams


def chrf_stats(hyp, ref, char_order=6, word_order=2):
    """Flattened [hyp, ref, match] counts for every char then word n-gram order."""
    stats = []
    for h, r in zip(_chrf_ngrams(hyp, char_order, word_order),
                    _chrf_ngrams(ref, char_order, word_order)):
        match = sum(min(c, r[g]) for g, c in h.items() if g in r)
        stats += [sum(h.values()) if r else 0, sum(r.values()), match]
    return stats


def chrf_from_stats(stats, beta=2):
    factor = beta ** 2
    avg_p = avg_r = 0.0
    eff = 0
    for i in range(len(stats) // 3):
        n_hyp, n_ref, n_match = stats[3 * i:3 * i + 3]
        if n_hyp > 0 and n_ref > 0:
            avg_p += n_match / n_hyp
            avg_r += n_match / n_ref
            eff += 1
    if eff == 0:
        return 0.0
    avg_p, avg_r = avg_p / eff, avg_r / eff
    if avg_p + avg_r == 0:
        return 0.0
    return 100.0 * (1 + factor) * avg_p * avg_r / (factor * avg_p + avg_r)


# ---------------------------------------------------------------- public API
def _sum_stats(rows):
    return [sum(col) for col in zip(*rows)] if rows else None


def segment_stats(preds, refs):
    """Per-segment (bleu_stats, chrf_stats); lets bootstrap resample without recomputing n-grams."""
    preds = ['' if p is None else str(p) for p in preds]
    refs = ['' if r is None else str(r) for r in refs]
    assert len(preds) == len(refs), f"{len(preds)} predictions vs {len(refs)} references"
    return [bleu_stats(p, r) for p, r in zip(preds, refs)], [chrf_stats(p, r) for p, r in zip(preds, refs)]


def scores_from_segment_stats(bleu_rows, chrf_rows):
    bleu = bleu_from_stats(_sum_stats(bleu_rows))
    chrfpp = chrf_from_stats(_sum_stats(chrf_rows))
    return {"bleu": bleu, "chrfpp": chrfpp, "geo_mean": math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))}


def corpus_scores(preds, refs):
    """Corpus BLEU, chrF++ and their geometric mean (the competition score), all on 0-100."""
    return scores_from_segment_stats(*segment_stats(preds, refs))


def sentence_chrfpp(hyp, ref):
    """Sentence-level chrF++, used as the MBR utility."""
    return chrf_from_stats(chrf_stats(hyp, ref))


METRIC_SIGNATURE = "BLEU=sacrebleu-2.x-13a-exp | chrF++=sacrebleu-2.x-c6-w2-b2 | geo_mean=sqrt(BLEU*chrF++)"

In [5]:
# >>> SHARED CELL: common/runlog.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import datetime
import json
import os

import pandas as pd


RESULTS_DIR = "results"


def finalize_run(system, seed, ids, refs, preds, hyperparameters, log_history=None,
                 out_dir=RESULTS_DIR, notes=None):
    os.makedirs(out_dir, exist_ok=True)
    ids = [str(i) for i in ids]
    refs = ["" if r is None else str(r) for r in refs]
    preds = ["" if p is None else str(p) for p in preds]
    scores = corpus_scores(preds, refs)
    tag = f"{system}__seed{seed}"
    pred_path = os.path.join(out_dir, f"preds_val_{tag}.csv")
    pd.DataFrame({"id": ids, "target_text": refs, "pred_translation": preds}).to_csv(pred_path, index=False)
    record = {
        "system": system,
        "seed": seed,
        "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "metric": METRIC_SIGNATURE,
        "n_val_docs": len(ids),
        "val_ids_sha256": val_ids_sha256(ids),
        "final_val": scores,
        "hyperparameters": hyperparameters,
        "notes": notes,
        "log_history": log_history or [],
    }
    with open(os.path.join(out_dir, f"run_{tag}.json"), "w") as f:
        json.dump(record, f, indent=2, default=str, ensure_ascii=False)
    print(f"[run record] {tag}: BLEU={scores['bleu']:.2f} chrF++={scores['chrfpp']:.2f} "
          f"GM={scores['geo_mean']:.2f} -> {pred_path}")
    return scores

In [6]:
# >>> SHARED CELL: common/env.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import importlib
import json
import os
import platform
import sys

TRACKED_PACKAGES = ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes",
                    "sentencepiece", "sacrebleu", "pandas", "numpy", "scikit-learn"]


def report_environment(notebook, out_dir="results"):
    versions = {}
    for name in TRACKED_PACKAGES:
        try:
            module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
            versions[name] = getattr(module, "__version__", "unknown")
        except Exception:
            versions[name] = None
    env = {"notebook": notebook, "python": sys.version.split()[0], "platform": platform.platform(),
           "packages": versions, "gpu": None, "cuda": None}
    try:
        import torch
        if torch.cuda.is_available():
            env["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
            env["cuda"] = torch.version.cuda
    except Exception:
        pass
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"env_{notebook}.json"), "w") as f:
        json.dump(env, f, indent=2)
    print(json.dumps(env, indent=2))
    return env

In [7]:
report_environment("qwen")  # -> results/env_qwen.json

{
  "notebook": "qwen",
  "python": "3.13.16",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "datasets": "4.8.5",
    "accelerate": "1.15.0",
    "peft": "0.21.2",
    "bitsandbytes": "0.50.2",
    "sentencepiece": "0.2.2",
    "sacrebleu": "2.6.0",
    "pandas": "2.3.3",
    "numpy": "2.1.3",
    "scikit-learn": "1.6.1"
  },
  "gpu": [
    "Tesla T4",
    "Tesla T4"
  ],
  "cuda": "13.0"
}


{'notebook': 'qwen',
 'python': '3.13.16',
 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39',
 'packages': {'torch': '2.11.0+cu130',
  'transformers': '5.20.2',
  'datasets': '4.8.5',
  'accelerate': '1.15.0',
  'peft': '0.21.2',
  'bitsandbytes': '0.50.2',
  'sentencepiece': '0.2.2',
  'sacrebleu': '2.6.0',
  'pandas': '2.3.3',
  'numpy': '2.1.3',
  'scikit-learn': '1.6.1'},
 'gpu': ['Tesla T4', 'Tesla T4'],
 'cuda': '13.0'}

In [8]:
import argparse
import json
import os
import random
from contextlib import nullcontext
from pathlib import Path

#EXPERIMENT SETTINGS

# True:
#   - Read Kaggle Secrets
#   - Use MLflow / DagsHub
#   - Upload LoRA adapter to Hugging Face
#
# False:
#   - Do NOT read any Secrets
#   - Do NOT use MLflow / DagsHub
#   - Do NOT upload to Hugging Face
#   - Save the LoRA adapter locally
#
# For reproducibility use FALSE and turn on any GPU environment

USE_TRACKING = False

# transformers v5 removed `warmup_ratio`: `warmup_steps` as a float < 1 is the ratio.
import transformers

WARMUP_KWARGS = (
    {"warmup_steps": 0.05}
    if int(transformers.__version__.split(".")[0]) >= 5
    else {"warmup_ratio": 0.05}
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import mlflow
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)


#PROMPT

SYSTEM_PROMPT = (
    "You are an expert translator of Akkadian. "
    "Translate the Akkadian transliteration into accurate English. "
    "Return only the English translation."
)


def build_prompt(source: str) -> str:
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Akkadian:\n{source}\n\n"
        f"English:\n"
    )


#TOKENIZATION

def tokenize_example(example, tokenizer, max_length):
    prompt = build_prompt(example["source_text"])
    target = example["target_text"]

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    eos = (
        [tokenizer.eos_token_id]
        if tokenizer.eos_token_id is not None
        else []
    )

    available = max_length - len(eos)

    if available <= 0:
        raise ValueError(
            f"max_length={max_length} is too small to fit EOS."
        )

    # Priority:
    # 1. Preserve the prompt.
    # 2. Use remaining tokens for the target.

    if len(prompt_ids) >= available:
        prompt_ids = prompt_ids[:available]
        target_ids = []
    else:
        remaining = available - len(prompt_ids)
        target_ids = target_ids[:remaining]

    input_ids = prompt_ids + target_ids + eos

    labels = (
            [-100] * len(prompt_ids)
            + target_ids
            + eos
    )

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


#DATASET

class ListDataset(torch.utils.data.Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]


#GENERATION

def generate_predictions(
        model,
        tokenizer,
        df,
        max_input_length,
        max_new_tokens,
        batch_size,
):

    model.eval()

    predictions = []

    # Disable generation sampling parameters inherited from
    # the model's generation_config.
    if hasattr(model, "generation_config"):
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

    # Left padding so that every prompt ends right before the generated tokens.
    tokenizer.padding_side = "left"
    for start in range(0, len(df), batch_size):

        batch = df.iloc[start:start + batch_size]

        prompts = [
            build_prompt(source)
            for source in batch["source_text"]
        ]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )

        enc = {
            key: value.to(model.device)
            for key, value in enc.items()
        }

        with torch.no_grad():

            outputs = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, sequence in enumerate(outputs):

            # With left padding every row of the batch has the same prompt width.
            prompt_len = enc["input_ids"].shape[1]

            generated_tokens = sequence[prompt_len:]

            text = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
            ).strip()

            predictions.append(text)

    return predictions


#LENGTH STATISTICS

def print_length_statistics(dataset):

    lengths = [
        len(example["input_ids"])
        for example in dataset
    ]

    if not lengths:
        print("Dataset is empty.")
        return

    print("\n" + "=" * 50)
    print("TOKEN LENGTH STATISTICS")
    print("=" * 50)

    print(f"Max:  {max(lengths)}")
    print(f"Mean: {sum(lengths) / len(lengths):.2f}")

    sorted_lengths = sorted(lengths)

    for percentile in [50, 75, 90, 95, 99]:

        index = min(
            int(len(sorted_lengths) * percentile / 100),
            len(sorted_lengths) - 1,
        )

        value = sorted_lengths[index]

        print(f"P{percentile}: {value}")

    print("=" * 50)


#MAIN

def main():

    parser = argparse.ArgumentParser()

    #Model / data
    parser.add_argument(
        "--model",
        default="Qwen/Qwen2.5-7B-Instruct",
    )

    parser.add_argument(
        "--train",
        default=(
            "/kaggle/input/competitions/"
            "deep-past-initiative-machine-translation/train.csv"
        ),
    )

    parser.add_argument(
        "--val",
        default=(
            "/kaggle/input/competitions/"
            "deep-past-initiative-machine-translation/test.csv"
        ),
    )

    parser.add_argument(
        "--output-dir",
        default="outputs/qwen2.5-7b-lora",
    )

    #Sequence / generation
    parser.add_argument(
        "--max-length",
        type=int,
        default=1024,  # prompt + source + target; p95 of source+target = 950 Qwen tokens (was 512)
    )

    parser.add_argument(
        "--max-new-tokens",
        type=int,
        default=448,  # p95 of translations = 419 Qwen tokens (was 256)
    )

    #Training
    parser.add_argument(
        "--epochs",
        type=float,
        default=2.0,
    )

    parser.add_argument(
        "--learning-rate",
        type=float,
        default=2e-4,
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=1,
    )

    parser.add_argument(
        "--grad-accum",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--eval-batch-size",
        type=int,
        default=1,
    )

    #LoRA
    parser.add_argument(
        "--lora-r",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--lora-alpha",
        type=int,
        default=32,
    )

    parser.add_argument(
        "--lora-dropout",
        type=float,
        default=0.05,
    )

    #Misc
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )

    args, unknown = parser.parse_known_args()

    #ENVIRONMENT
    load_dotenv()

    set_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)

    # KAGGLE SECRETS
    # Secrets are read ONLY when USE_TRACKING=True.
    hf_token = None
    hf_repo = None

    if USE_TRACKING:

        print("\nLoading Kaggle Secrets...")

        secrets = UserSecretsClient()

        os.environ["HF_TOKEN"] = secrets.get_secret(
            "HF_TOKEN"
        )

        os.environ["HF_REPO_ID"] = secrets.get_secret(
            "HF_REPO_ID"
        )

        os.environ["DAGSHUB_USERNAME"] = secrets.get_secret(
            "DAGSHUB_USERNAME"
        )

        os.environ["DAGSHUB_TOKEN"] = secrets.get_secret(
            "DAGSHUB_TOKEN"
        )

        os.environ["DAGSHUB_PASSWORD"] = secrets.get_secret(
            "DAGSHUB_PASSWORD"
        )

        os.environ["MLFLOW_TRACKING_URI"] = secrets.get_secret(
            "MLFLOW_TRACKING_URI"
        )

        os.environ["MLFLOW_EXPERIMENT_NAME"] = secrets.get_secret(
            "MLFLOW_EXPERIMENT_NAME"
        )

        #DagsHub credentials for MLflow
        os.environ["MLFLOW_TRACKING_USERNAME"] = (
            os.environ["DAGSHUB_USERNAME"]
        )

        os.environ["MLFLOW_TRACKING_PASSWORD"] = (
            os.environ["DAGSHUB_TOKEN"]
        )

        hf_token = os.getenv("HF_TOKEN")
        hf_repo = os.getenv("HF_REPO_ID")

    #DEVICE
    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA requires a CUDA-capable GPU with bitsandbytes."
        )

    print("\n" + "=" * 60)
    print("DEVICE")
    print("=" * 60)

    print(
        "CUDA available:",
        torch.cuda.is_available(),
    )

    print(
        "CUDA devices:",
        torch.cuda.device_count(),
    )

    print(
        "Device:",
        torch.cuda.get_device_name(0),
    )

    print(
        "Tracking:",
        USE_TRACKING,
    )

    print("=" * 60)

    #AUTH

    if USE_TRACKING:

        if not hf_token:
            raise RuntimeError(
                "HF_TOKEN is missing."
            )

        login(token=hf_token)

    #DATA
    print("\nLoading dataset...")

    train_df = pd.read_csv(args.train)

    # Shared document-level split (identical in every notebook, verified by checksum).
    # Before Phase 2 this notebook split rows with train_test_split.
    train_df, val_df = make_split(train_df)
    val_ids_raw = val_df["oare_id"].astype(str).tolist()
    val_refs_raw = val_df["translation"].astype(str).tolist()

    print(f"Full dataset size: {len(train_df)}")

    train_df = train_df.rename(
        columns={
            "transliteration": "source_text",
            "translation": "target_text",
        }
    )

    # Keep only valid examples
    train_df["source_text"] = train_df["source_text"].astype(str).str.strip()
    train_df["target_text"] = train_df["target_text"].astype(str).str.strip()

    train_df = train_df[
        (train_df["source_text"] != "")
        & (train_df["target_text"] != "")
        ].reset_index(drop=True)

    #SPLIT: done above by make_split; the validation rows are only renamed, never filtered.
    val_df = val_df.rename(columns={"transliteration": "source_text", "translation": "target_text"})
    val_df["source_text"] = val_df["source_text"].astype(str).str.strip()
    val_df["target_text"] = val_df["target_text"].astype(str).str.strip()

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(f"Train size: {len(train_df)}")
    print(f"Val size:   {len(val_df)}")

    #Debugging
    required_columns = [
        "source_text",
        "target_text",
    ]

    for column in required_columns:

        if column not in train_df.columns:
            raise ValueError(
                f"Missing column '{column}' in train dataset."
            )

        if column not in val_df.columns:
            raise ValueError(
                f"Missing column '{column}' in validation dataset."
            )

    #TOKENIZER
    print("\nLoading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        args.model,
        token=hf_token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    #QUANTIZATION
    print("\nCreating 4-bit quantization config...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    #MODEL
    print("\nLoading model...")

    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        quantization_config=bnb_config,

        # IMPORTANT:
        # Do NOT use device_map="auto".
        device_map={"": 0},

        dtype=torch.float16,
        token=hf_token,
    )

    model.config.use_cache = False

    # Required for QLoRA.
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    # prepare_model_for_kbit_training upcasts ALL non-quantized params to fp32,
    # including embed_tokens and lm_head (~1.1B params for Qwen2.5-7B, vocab 152k).
    # That wastes ~2.2 GB on a 15 GB T4 and caused the CUDA OOM in backward.
    # Cast them back to fp16 (LoRA adapters do not touch these layers).
    model.get_input_embeddings().to(torch.float16)
    model.get_output_embeddings().to(torch.float16)

    #LoRA

    peft_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",

        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = get_peft_model(
        model,
        peft_config,
    )

    print("\nTrainable parameters:")

    model.print_trainable_parameters()

    #TOKENIZE DATA
    print("\nTokenizing train dataset...")

    train_records = [
        tokenize_example(
            row,
            tokenizer,
            args.max_length,
        )
        for row in train_df.to_dict("records")
    ]

    print("Tokenizing validation dataset...")

    val_records = [
        tokenize_example(
            row,
            tokenizer,
            args.max_length,
        )
        for row in val_df.to_dict("records")
    ]

    train_dataset = ListDataset(train_records)
    val_dataset = ListDataset(val_records)

    print_length_statistics(train_dataset)

    #COLLATOR
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )

    #OUTPUT
    output_dir = Path(args.output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    #TRAINING ARGUMENTS

    training_args = TrainingArguments(
        output_dir=str(output_dir),

        #Training duration
        num_train_epochs=args.epochs,

        #Optimization
        learning_rate=args.learning_rate,

        lr_scheduler_type="cosine",

        **WARMUP_KWARGS,  # 5% warmup (warmup_ratio was removed in transformers v5)

        weight_decay=0.01,

        optim="paged_adamw_8bit",

        #Batch

        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,

        #Precision

        fp16=True,
        bf16=False,

        #Memory

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        #Evaluation
        eval_strategy="epoch",

        #Checkpoints
        save_strategy="epoch",

        save_total_limit=2,

        # VERY IMPORTANT:
        # After training, Trainer restores the checkpoint
        # with the lowest eval_loss.

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        #Logging
        logging_steps=10,

        #Do not send Trainer logs to external integrations.
        report_to=[],

        #Reproducibility
        seed=args.seed,
        data_seed=args.seed,
    )

    #MLFLOW SETUP
    if USE_TRACKING:

        mlflow.set_tracking_uri(
            os.getenv("MLFLOW_TRACKING_URI")
        )

        mlflow.set_experiment(
            os.getenv(
                "MLFLOW_EXPERIMENT_NAME",
                "akkadian-qwen2.5-lora",
            )
        )

    #START RUN
    run_context = (
        mlflow.start_run()
        if USE_TRACKING
        else nullcontext(None)
    )

    with run_context as run:

        #MLFLOW PARAMETERS
        if USE_TRACKING:

            mlflow.log_params({
                "base_model": args.model,

                "architecture":
                    "qwen2.5-7b-qlora",

                "learning_rate":
                    args.learning_rate,

                "lr_scheduler":
                    "cosine",

                "warmup_ratio":
                    0.05,

                "weight_decay":
                    0.01,

                "batch_size":
                    args.batch_size,

                "gradient_accumulation":
                    args.grad_accum,

                "max_length":
                    args.max_length,

                "max_new_tokens":
                    args.max_new_tokens,

                "epochs":
                    args.epochs,

                "lora_r":
                    args.lora_r,

                "lora_alpha":
                    args.lora_alpha,

                "lora_dropout":
                    args.lora_dropout,

                "quantization":
                    "4bit-nf4-double-quant",

                "compute_dtype":
                    "float16",

                "gradient_checkpointing":
                    True,

                "optimizer":
                    "paged_adamw_8bit",

                "train_size":
                    len(train_df),

                "val_size":
                    len(val_df),

                "seed":
                    args.seed,
            })

        #TRAINER

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=collator,
        )

        #TRAIN
        print("\n" + "=" * 60)
        print("START TRAINING")
        print("=" * 60)

        train_result = trainer.train()

        #EVALUATION
        print("\n" + "=" * 60)
        print("FINAL EVALUATION")
        print("=" * 60)

        eval_metrics = trainer.evaluate()

        train_metrics = train_result.metrics

        #LOG LOSSES
        if USE_TRACKING:

            if "train_loss" in train_metrics:

                mlflow.log_metric(
                    "train_loss",
                    float(train_metrics["train_loss"]),
                )

            if "eval_loss" in eval_metrics:

                mlflow.log_metric(
                    "val_loss",
                    float(eval_metrics["eval_loss"]),
                )

        #SAVE BEST MODEL
        print("\nSaving best model...")

        trainer.save_model(
            str(output_dir)
        )

        tokenizer.save_pretrained(
            str(output_dir)
        )


        #GENERATION
        print("\n" + "=" * 60)
        print("GENERATING VALIDATION PREDICTIONS")
        print("=" * 60)

        predictions = generate_predictions(
            model=model,
            tokenizer=tokenizer,
            df=val_df,
            max_input_length=args.max_length,
            max_new_tokens=args.max_new_tokens,
            batch_size=args.eval_batch_size,
        )

        references = (
            val_df["target_text"]
            .astype(str)
            .tolist()
        )

        # Shared metric (sacreBLEU-identical BLEU + chrF++), full untruncated references
        _scores = corpus_scores(predictions, references)
        bleu, chrfpp, geom_mean = _scores["bleu"], _scores["chrfpp"], _scores["geo_mean"]

        #METRICS
        print("\n" + "=" * 60)
        print("TRANSLATION METRICS")
        print("=" * 60)

        print(
            f"BLEU:  {bleu:.4f}"
        )

        print(
            f"chrF++: {chrfpp:.4f}"
        )

        print(
            f"Geometric mean: {geom_mean}"
        )


        if USE_TRACKING:

            mlflow.log_metric(
                "val_BLEU",
                float(bleu),
            )

            mlflow.log_metric(
                "val_chrf_plus_plus",
                float(chrfpp),
            )

            mlflow.log_metric(
                "geom_mean_score",
                geom_mean,
            )

        #RUN RECORD (results/preds_val_qwen2.5-7b-lora__seed<k>.csv + run_*.json)
        finalize_run(
            system="qwen2.5-7b-lora", seed=args.seed, ids=val_ids_raw, refs=val_refs_raw,
            preds=predictions,
            hyperparameters={k: v for k, v in vars(args).items() if k not in ("train", "val")},
            log_history=trainer.state.log_history,
            notes="checkpoint selected by eval_loss (generation per epoch is too slow for a 7B model)",
        )

        #SAVE PREDICTIONS

        pred_df = val_df.copy()

        pred_df["pred_translation"] = predictions

        pred_path = (
                output_dir
                / "preds_val_qwen2.5-7b-lora.csv"
        )

        if "oare_id" in pred_df.columns:

            pred_df.rename(
                columns={
                    "oare_id": "id"
                },
                inplace=True,
            )

        pred_df.to_csv(
            pred_path,
            index=False,
        )

        if USE_TRACKING:

            mlflow.log_artifact(
                str(pred_path),
                artifact_path="predictions",
            )

        #HUGGING FACE
        if USE_TRACKING and hf_repo:

            print("\n" + "=" * 60)
            print("UPLOADING LoRA ADAPTER TO HUGGING FACE")
            print("=" * 60)

            # `model` now contains the best checkpoint.
            model.push_to_hub(
                hf_repo,
                private=True,
                token=hf_token,
                commit_message=(
                    "Upload best Akkadian QLoRA adapter"
                ),
            )

            tokenizer.push_to_hub(
                hf_repo,
                private=True,
                token=hf_token,
                commit_message=(
                    "Upload tokenizer"
                ),
            )

            mlflow.set_tag(
                "hf_repo",
                hf_repo,
            )


        #MLFLOW TAGS
        if USE_TRACKING:

            mlflow.set_tag(
                "model_architecture",
                "qwen2.5-lora",
            )

            mlflow.set_tag(
                "best_model_metric",
                "eval_loss",
            )

        #RUN SUMMARY
        summary = {
            "run_id": (
                run.info.run_id
                if USE_TRACKING and run is not None
                else None
            ),

            "base_model":
                args.model,

            "train_loss":
                train_metrics.get(
                    "train_loss"
                ),

            "val_loss":
                eval_metrics.get(
                    "eval_loss"
                ),

            "val_BLEU":
                float(bleu),

            "val_chrF++":
                float(chrfpp),

            "geom_mean_score":
                geom_mean,

            "epochs":
                args.epochs,

            "learning_rate":
                args.learning_rate,

            "lora_r":
                args.lora_r,

            "lora_alpha":
                args.lora_alpha,

            "max_length":
                args.max_length,

            "train_size":
                len(train_df),

            "val_size":
                len(val_df),

            "hf_repo":
                hf_repo if USE_TRACKING else None,
        }

        if USE_TRACKING:

            mlflow.log_dict(
                summary,
                "run_summary.json",
            )

        #FINAL OUTPUT
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE")
        print("=" * 60)

        print(
            json.dumps(
                summary,
                indent=2,
                ensure_ascii=False,
            )
        )

        print("\nLocal model saved to:")
        print(output_dir)

        if USE_TRACKING:

            print("\nHugging Face repository:")
            print(hf_repo)

        else:

            print("\nTracking disabled.")
            print("No MLflow/DagsHub logging.")
            print("No Hugging Face upload.")


if __name__ == "__main__":
    main()


DEVICE
CUDA available: True
CUDA devices: 2
Device: Tesla T4
Tracking: False

Loading dataset...
[shared split] train=1405 rows, val=156 rows (156 documents), val_ids_sha256=1939161cfcb9ff85...
Full dataset size: 1405
Train size: 1405
Val size:   156

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (8 outputs left out)


FINAL EVALUATION


Training Loss,Validation Loss,Epoch
2.356934,2.314205,2



Saving best model...

GENERATING VALIDATION PREDICTIONS


RuntimeError: expected scalar type Float but found Half

In [9]:
import sys, json, torch

# 1. достаём локальные переменные из упавшего main()
tb, frame = sys.last_traceback, None
while tb is not None:
    if tb.tb_frame.f_code.co_name == "main":
        frame = tb.tb_frame
        break
    tb = tb.tb_next
assert frame is not None, "traceback не найден, напиши мне, сделаю вариант с загрузкой адаптера с диска"
L = frame.f_locals
model, tokenizer, val_df, args = L["model"], L["tokenizer"], L["val_df"], L["args"]
trainer, output_dir = L["trainer"], L["output_dir"]
val_ids_raw, val_refs_raw = L["val_ids_raw"], L["val_refs_raw"]

# 2. генерация внутри autocast (старая generate_predictions без него падает)
with torch.autocast("cuda", dtype=torch.float16):
    predictions = generate_predictions(
        model=model, tokenizer=tokenizer, df=val_df,
        max_input_length=args.max_length,
        max_new_tokens=args.max_new_tokens,
        batch_size=args.eval_batch_size,   # можно поставить 4 для ускорения
    )

# 3. метрики и запись результатов, как в конце main()
references = val_df["target_text"].astype(str).tolist()
_scores = corpus_scores(predictions, references)
bleu, chrfpp, geom_mean = _scores["bleu"], _scores["chrfpp"], _scores["geo_mean"]
print(f"BLEU: {bleu:.4f} | chrF++: {chrfpp:.4f} | geo-mean: {geom_mean}")

finalize_run(
    system="qwen2.5-7b-lora", seed=args.seed, ids=val_ids_raw, refs=val_refs_raw,
    preds=predictions,
    hyperparameters={k: v for k, v in vars(args).items() if k not in ("train", "val")},
    log_history=trainer.state.log_history,
    notes="checkpoint selected by eval_loss (generation per epoch is too slow for a 7B model)",
)

pred_df = val_df.copy()
pred_df["pred_translation"] = predictions
if "oare_id" in pred_df.columns:
    pred_df.rename(columns={"oare_id": "id"}, inplace=True)
pred_df.to_csv(output_dir / "preds_val_qwen2.5-7b-lora.csv", index=False)
print("saved")

BLEU: 25.7279 | chrF++: 47.5632 | geo-mean: 34.98147005455389
[run record] qwen2.5-7b-lora__seed42: BLEU=25.73 chrF++=47.56 GM=34.98 -> results/preds_val_qwen2.5-7b-lora__seed42.csv
saved


/tmp/ipykernel_51/1224256274.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),


In [10]:
import shutil, os
from IPython.display import FileLink

src_adapter = "/kaggle/working/outputs/qwen2.5-7b-lora"
export = "/kaggle/working/export"
shutil.rmtree(export, ignore_errors=True)

# адаптер + токенизатор + предсказания, без checkpoint-*
shutil.copytree(src_adapter, f"{export}/qwen2.5-7b-lora",
                ignore=shutil.ignore_patterns("checkpoint-*"))
# метрики и предсказания прогона
shutil.copytree("/kaggle/working/results", f"{export}/results")

shutil.make_archive("/kaggle/working/qwen_lora_export", "zip", export)
print(os.path.getsize("/kaggle/working/qwen_lora_export.zip") / 1e6, "MB")
FileLink("qwen_lora_export.zip")

151.845046 MB


/kaggle/working/qwen_lora_export.zip

# Saving my model to Kaggle

In [ ]:
# ============================================================
# Upload trained LoRA adapter → Kaggle Models (only with PUBLISH_MODELS = True)
# ============================================================
PUBLISH_MODELS = True     # True: upload the adapter to Kaggle Models (needs Internet ON)
KAGGLE_USERNAME = "ruzhbaykaa"  # owner of the Kaggle Models read by ensemble-pipeline.ipynb
QWEN_SEED = 42             # the --seed used for training

if not PUBLISH_MODELS:
    print("PUBLISH_MODELS = False: adapter kept locally in outputs/qwen2.5-7b-lora, no upload.")
else:
    import os
    from pathlib import Path
    import kagglehub

    # ---------- settings ----------
    LOCAL_DIR       = Path("outputs/qwen2.5-7b-lora")   # куда сохранил trainer.save_model()
    FRAMEWORK       = "pytorch"
    MODEL_SLUG      = "akkadika"
    VARIATION       = f"qwen-lora-seed{QWEN_SEED}"

    handle = f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

    # ---------- sanity check ----------
    assert LOCAL_DIR.exists(), f"Не найдено: {LOCAL_DIR}. Сначала запусти обучение."
    print("Files to upload:", sorted(p.name for p in LOCAL_DIR.iterdir()))

    # ---------- upload ----------
    handle_used = kagglehub.model_upload(
        handle=handle,
        local_model_dir=str(LOCAL_DIR),
        version_notes=(
            "LoRA adapter for Qwen2.5-7B — Akkadika (Akkadian fine-tune). "
            "Requires base Qwen/Qwen2.5-7B-Instruct "
        ),
        license_name="Apache 2.0",
    )
    print("✅ Uploaded:", handle_used)